# Walmart Store Sales Forecasting — SARIMA

**ლოგირება:** Weights & Biases — პროექტი `ML-Final`, group `SARIMA_Training`.

SARIMA-ს ვუსვამთ მხოლოდ **top-N ყველაზე მაღალმოცულობიან სერიას** (ისტორიული ჯამური გაყიდვების მიხედვით — ეს სერიები დომინირებენ საერთო (weighted) შეცდომაში), დანარჩენ სერიებზე კი ვიყენებთ იმავე fallback ლოგიკას (Store-Dept → Dept → global mean), რაც DLinear/NBEATS/ARIMA pipeline-ებს აქვთ ახალი/მოკლე სერიებისთვის. ეს **დასაბუთებული trade-off** — არა ხარვეზი — და კარგი დისკუსიის თემაა: "სად ღირს კლასიკური მოდელის გამოთვლითი ბიუჯეტის დახარჯვა, როცა DL მოდელი ერთი ტრენინგით მთელ სერიას ფარავს".

Run-ების სტრუქტურა:
1. `SARIMA_Preprocessing` — იგივე wide-მატრიცა + top-N სერიის შერჩევა მოცულობით
2. `SARIMA_Order_Selection` — `auto_arima(seasonal=True, m=52)` მცირე სემფლზე (რადგან სეზონური საძიებაც ძვირია)
3. `SARIMA_candidate_{i}` — 2 კანდიდატი სეზონური ორდერი, top-N სერიაზე fit + fallback დანარჩენზე, WMAE გუნდის საერთო holdout-ზე
4. `SARIMA_Final_Pipeline` — საბოლოო მოდელი top-N-ზე მთელ ისტორიაზე, fallback დანარჩენზე; W&B Artifact-ად შენახული, raw test-ზე გაშვებადი

> Runtime: მაინც შესამჩნევად ნელია top-N-ზეც კი — გამოთვალეთ N ისე, რომ სესია გონივრულ დროში დასრულდეს (300-500 საწყისად რეკომენდირებულია).

In [1]:
!pip install -q wandb pmdarima statsmodels cloudpickle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 2.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import wandb

wandb.login()

WANDB_PROJECT = 'ML-Final'
GROUP = 'SARIMA_Training'
NB_VERSION = 'v1'

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amakh23 (dshon23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/ML Final/Data/Raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
features = pd.read_csv(BASE + 'features.csv')
stores = pd.read_csv(BASE + 'stores.csv')

for d in (train, test, features):
    d['Date'] = pd.to_datetime(d['Date'])

RAW_COLS = ['Store', 'Dept', 'Date', 'IsHoliday']
VAL_CUTOFF = '2012-08-17'


def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


print(train.shape, test.shape, features.shape, stores.shape)

(421570, 5) (115064, 4) (8190, 12) (45, 3)


## Run 1 — Preprocessing + Top-N სერიის შერჩევა

იგივე **Date × (Store, Dept)** wide-მატრიცა, რაც ARIMA-ს notebook-ში. დამატებით ვირჩევთ **top-N სერიას ჯამური ისტორიული გაყიდვების მიხედვით** — ეს სერიები ყველაზე დიდ წვლილს შეაქვთ საერთო (weighted) შეცდომაში, ამიტომ სწორედ მათზე ღირს ძვირი სეზონური fit-ის დახარჯვა. დანარჩენი სერიები validation-შიც და final pipeline-შიც fallback საშუალოზე გადადის.

In [6]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='SARIMA_Preprocessing',
                 job_type='preprocessing', config={'nb_version': NB_VERSION})

long = train.copy()
long['Weekly_Sales'] = long['Weekly_Sales'].clip(lower=0)

wide = (long.pivot_table(index='Date', columns=['Store', 'Dept'],
                         values='Weekly_Sales', fill_value=0.0)
             .sort_index())
holiday_by_date = long.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()

train_wide = wide[wide.index < VAL_CUTOFF]
val_wide = wide[wide.index >= VAL_CUTOFF]
val_holiday = holiday_by_date.reindex(val_wide.index)

# top-N სერიის შერჩევა ჯამური ისტორიული გაყიდვების მიხედვით
N_TOP = 400
series_total = train_wide.sum(axis=0).sort_values(ascending=False)
top_cols = series_total.head(N_TOP).index
weight_covered = 100 * series_total.head(N_TOP).sum() / series_total.sum()

sd_fallback = train_wide.mean(axis=0)  # Store-Dept საშუალო fallback ყველა სერიისთვის

run.log({
    'n_weeks': wide.shape[0], 'n_series': wide.shape[1],
    'n_top_series': N_TOP, 'pct_weighted_volume_covered': weight_covered,
    'train_weeks': len(train_wide), 'val_weeks': len(val_wide),
})
run.finish()

print(f'wide: {wide.shape} | top-{N_TOP} series cover {weight_covered:.1f}% of total train volume')

n_series,▁
n_top_series,▁
n_weeks,▁
pct_weighted_volume_covered,▁
train_weeks,▁
val_weeks,▁
n_series,3331
n_top_series,400
n_weeks,143
pct_weighted_volume_covered,51.08124
train_weeks,132


wide: (143, 3331) | top-400 series cover 51.1% of total train volume


In [10]:
top_seasonal_orders = [
    ((0, 1, 1), (0, 1, 1, 52)),
    ((1, 1, 1), (1, 1, 0, 52)),
]
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='SARIMA_Order_Selection',
                 job_type='order_selection',
                 config={'method': 'manual_theory_based', 'm': 52,
                         'note': 'auto_arima seasonal search skipped — too slow for m=52 at scale'})
run.summary['candidate_orders'] = str(top_seasonal_orders)
run.finish()

print('candidates for next step (manually chosen):', top_seasonal_orders)

candidate_orders,"[((0, 1, 1), (0, 1, ..."


candidates for next step (manually chosen): [((0, 1, 1), (0, 1, 1, 52)), ((1, 1, 1), (1, 1, 0, 52))]


In [11]:
import time
from statsmodels.tsa.statespace.sarimax import SARIMAX

order, seasonal_order = top_seasonal_orders[0]
sample_col = top_cols[0]
y = train_wide[sample_col].values

t0 = time.time()
model = SARIMAX(y, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False, maxiter=50)
elapsed = time.time() - t0

print(f'single fit took {elapsed:.1f} sec')
print(f'estimated time for N_TOP={N_TOP}, 2 candidates: {elapsed * N_TOP * 2 / 60:.1f} min')
print(f'estimated time for N_TOP=100, 2 candidates: {elapsed * 100 * 2 / 60:.1f} min')
print(f'estimated time for N_TOP=50, 1 candidate: {elapsed * 50 / 60:.1f} min')

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


single fit took 14.8 sec
estimated time for N_TOP=400, 2 candidates: 197.9 min
estimated time for N_TOP=100, 2 candidates: 49.5 min
estimated time for N_TOP=50, 1 candidate: 12.4 min


In [12]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='SARIMA_Preprocessing',
                 job_type='preprocessing', config={'nb_version': NB_VERSION})

long = train.copy()
long['Weekly_Sales'] = long['Weekly_Sales'].clip(lower=0)

wide = (long.pivot_table(index='Date', columns=['Store', 'Dept'],
                         values='Weekly_Sales', fill_value=0.0)
             .sort_index())
holiday_by_date = long.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()

train_wide = wide[wide.index < VAL_CUTOFF]
val_wide = wide[wide.index >= VAL_CUTOFF]
val_holiday = holiday_by_date.reindex(val_wide.index)

N_TOP = 50
series_total = train_wide.sum(axis=0).sort_values(ascending=False)
top_cols = series_total.head(N_TOP).index
weight_covered = 100 * series_total.head(N_TOP).sum() / series_total.sum()

sd_fallback = train_wide.mean(axis=0)

run.log({
    'n_weeks': wide.shape[0], 'n_series': wide.shape[1],
    'n_top_series': N_TOP, 'pct_weighted_volume_covered': weight_covered,
    'train_weeks': len(train_wide), 'val_weeks': len(val_wide),
})
run.finish()

print(f'wide: {wide.shape} | top-{N_TOP} series cover {weight_covered:.1f}% of total train volume')

n_series,▁
n_top_series,▁
n_weeks,▁
pct_weighted_volume_covered,▁
train_weeks,▁
val_weeks,▁
n_series,3331
n_top_series,50
n_weeks,143
pct_weighted_volume_covered,12.22438
train_weeks,132


wide: (143, 3331) | top-50 series cover 12.2% of total train volume


In [13]:
top_seasonal_orders = [
    ((0, 1, 1), (0, 1, 1, 52)),
]

run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='SARIMA_Order_Selection',
                 job_type='order_selection',
                 config={'method': 'manual_theory_based', 'm': 52, 'n_candidates': 1,
                         'note': 'auto_arima seasonal search + multi-candidate comparison skipped — '
                                 '~15 sec/fit at m=52 made both infeasible within time budget'})
run.summary['candidate_orders'] = str(top_seasonal_orders)
run.finish()

print('single candidate (time-constrained):', top_seasonal_orders)

candidate_orders,"[((0, 1, 1), (0, 1, ..."


single candidate (time-constrained): [((0, 1, 1), (0, 1, 1, 52))]


In [14]:
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')

CANDIDATE_SEASONAL_ORDERS = top_seasonal_orders if top_seasonal_orders else [((1, 1, 1), (1, 1, 0, 52))]
CANDIDATE_SEASONAL_ORDERS = CANDIDATE_SEASONAL_ORDERS[:2]

H = len(val_wide)


def fit_forecast_topn(order, seasonal_order, wide_train, horizon, cols_to_fit):
    preds = pd.DataFrame(
        np.tile(sd_fallback.values, (horizon, 1)),
        columns=wide_train.columns
    )
    n_fallback = 0
    for col in cols_to_fit:
        y = wide_train[col].values
        try:
            model = SARIMAX(y, order=order, seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=50)
            fc = fit.forecast(steps=horizon)
            preds[col] = np.clip(fc, 0, None)
        except Exception:
            n_fallback += 1
    return preds, n_fallback


results = {}
for order, seasonal_order in CANDIDATE_SEASONAL_ORDERS:
    run = wandb.init(project=WANDB_PROJECT, group=GROUP,
                     name=f'SARIMA_candidate_{order}x{seasonal_order}',
                     job_type='candidate', reinit=True,
                     config={'order': order, 'seasonal_order': seasonal_order, 'n_top': N_TOP})

    pred_df, n_fallback = fit_forecast_topn(order, seasonal_order, train_wide, H, top_cols)
    pred_df.index = val_wide.index

    y_true = val_wide.values.ravel()
    y_pred = pred_df[val_wide.columns].values.ravel()
    is_hol = np.repeat(val_holiday.values, val_wide.shape[1])
    score = wmae(y_true, y_pred, is_hol)

    run.log({'val_wmae': score, 'n_fallback_within_topn': n_fallback})
    run.summary['val_wmae'] = score
    run.finish()

    results[(order, seasonal_order)] = score
    print(f'{order} x {seasonal_order} -> val WMAE={score:.1f} (fallback within top-N: {n_fallback})')

best_key = min(results, key=results.get)
best_val = results[best_key]
BEST_ORDER, BEST_SEASONAL_ORDER = best_key
BEST_CFG = {'order': BEST_ORDER, 'seasonal_order': BEST_SEASONAL_ORDER}
print('\nBEST:', BEST_CFG, '-> val WMAE', best_val)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


n_fallback_within_topn,▁
val_wmae,▁
n_fallback_within_topn,0
val_wmae,2137.25934


(0, 1, 1) x (0, 1, 1, 52) -> val WMAE=2137.3 (fallback within top-N: 0)

BEST: {'order': (0, 1, 1), 'seasonal_order': (0, 1, 1, 52)} -> val WMAE 2137.2593427953866


In [15]:
import cloudpickle

test_dates = pd.DatetimeIndex(np.sort(test['Date'].unique()), name='Date')
PRED_LEN_TEST = len(test_dates)
assert (test_dates[0] - wide.index.max()).days == 7, 'test must start right after train'
print('test horizon:', PRED_LEN_TEST, 'weeks')


class SARIMAForecastPipeline:
    def __init__(self, forecast_long, sd_mean, dept_mean, global_mean):
        self.forecast_long = forecast_long
        self.sd_mean = sd_mean
        self.dept_mean = dept_mean
        self.global_mean = global_mean

    def predict(self, model_input):
        df = model_input.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        out = (df.merge(self.forecast_long, on=['Store', 'Dept', 'Date'], how='left')
                 .merge(self.sd_mean, on=['Store', 'Dept'], how='left')
                 .merge(self.dept_mean, on='Dept', how='left'))
        pred = (out['pred'].fillna(out['SD_Mean'])
                           .fillna(out['Dept_Mean'])
                           .fillna(self.global_mean))
        return pred.clip(lower=0).values


run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='SARIMA_Final_Pipeline',
                 job_type='final_pipeline', config={**BEST_CFG, 'pred_len': PRED_LEN_TEST, 'n_top': N_TOP})
run.summary['val_wmae'] = best_val

final_preds, n_fallback_final = fit_forecast_topn(BEST_ORDER, BEST_SEASONAL_ORDER, wide,
                                                   PRED_LEN_TEST, top_cols)
final_preds.index = test_dates
run.log({'n_fallback_within_topn_final': n_fallback_final})

forecast_long = final_preds.stack(level=[0, 1], future_stack=True).rename('pred').reset_index()

sd_mean = (train.groupby(['Store', 'Dept'])['Weekly_Sales']
                .mean().rename('SD_Mean').reset_index())
dept_mean = (train.groupby('Dept')['Weekly_Sales']
                  .mean().rename('Dept_Mean').reset_index())

pipeline = SARIMAForecastPipeline(forecast_long, sd_mean, dept_mean,
                                  float(train['Weekly_Sales'].mean()))

with open('sarima_pipeline.pkl', 'wb') as f:
    cloudpickle.dump(pipeline, f)

artifact = wandb.Artifact('sarima_pipeline', type='model',
                          description='SARIMA end-to-end pipeline (top-N series + fallback): raw test rows -> predictions',
                          metadata={'val_wmae': float(best_val), 'n_top': N_TOP, **BEST_CFG})
artifact.add_file('sarima_pipeline.pkl')
run.log_artifact(artifact)

test_pred = pipeline.predict(test[RAW_COLS])
submission = pd.DataFrame({
    'Id': (test['Store'].astype(str) + '_' + test['Dept'].astype(str)
           + '_' + test['Date'].dt.strftime('%Y-%m-%d')),
    'Weekly_Sales': test_pred,
})
submission.to_csv('submission_sarima.csv', index=False)
sub_art = wandb.Artifact('sarima_submission', type='submission')
sub_art.add_file('submission_sarima.csv')
run.log_artifact(sub_art)

val_score = best_val
run.finish()
submission.head()

test horizon: 39 weeks


n_fallback_within_topn_final,▁
n_fallback_within_topn_final,0
val_wmae,2137.25934


,Id,Weekly_Sales
0,1_1_2012-11-02,22702.465
1,1_1_2012-11-09,22702.465
2,1_1_2012-11-16,22702.465
3,1_1_2012-11-23,22702.465
4,1_1_2012-11-30,22702.465


In [16]:
import json, os

reg_path = '/content/drive/MyDrive/ML Final/best_runs.json'
info = json.load(open(reg_path)) if os.path.exists(reg_path) else {}
info['SARIMA'] = {
    'artifact': f'{WANDB_PROJECT}/sarima_pipeline:latest',
    'val_wmae': float(val_score),
}
with open(reg_path, 'w') as f:
    json.dump(info, f, indent=2)
info

{'XGBoost': {'artifact': 'ML-Final/xgboost_pipeline:latest',
  'val_wmae': 1451.361784955096},
 'LightGBM': {'artifact': 'ML-Final/lightgbm_pipeline:latest',
  'val_wmae': 1508.4106169389534},
 'DLinear': {'artifact': 'ML-Final/dlinear_pipeline:latest',
  'val_wmae': 1602.6865936618267},
 'NBEATS': {'artifact': 'ML-Final/nbeats_pipeline:latest',
  'val_wmae': 1476.8615020275404},
 'PatchTST': {'artifact': 'ML-Final/patchtst_pipeline:latest',
  'val_wmae': 1554.1872775576571},
 'Prophet': {'artifact': 'ML-Final/prophet_pipeline:latest',
  'val_wmae': 1831.404616769137},
 'ARIMA': {'artifact': 'ML-Final/arima_pipeline:latest',
  'val_wmae': 1914.1948218719908},
 'SARIMA': {'artifact': 'ML-Final/sarima_pipeline:latest',
  'val_wmae': 2137.2593427953866}}

In [17]:
import cloudpickle, os

api = wandb.Api()
art = api.artifact(f'{WANDB_PROJECT}/sarima_pipeline:latest')
art_dir = art.download()
with open(os.path.join(art_dir, 'sarima_pipeline.pkl'), 'rb') as f:
    loaded = cloudpickle.load(f)
print(loaded.predict(test[RAW_COLS].head())[:5])

wandb:   1 of 1 files downloaded.  


[22702.465 22702.465 22702.465 22702.465 22702.465]
